In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ==========================================
# 1. MATERIAL & CORE PATTERN DEFINITIONS
# ==========================================
FACE_MATERIALS = {
    "Aluminum 6061-T6": {"E": 68.9e9, "rho": 2700.0},
    "Carbon Fiber (UD)":  {"E": 135.0e9, "rho": 1600.0},
    "PLA (3D Print)":     {"E": 2.8e9,  "rho": 1240.0},
}

CORE_PATTERNS = {
    "Honeycomb": {
        "rel_density_factor": 0.045,
        "shear_mod_factor":   0.040,
        "note": "Hexagonal cells provide high bending stiffness-to-weight and shear resistance with low relative density."
    },
    "Corrugated (Wave)": {
        "rel_density_factor": 0.065,
        "shear_mod_factor":   0.032,
        "note": "Wavy/corrugated core increases out-of-plane stiffness cleanly; directional stiffness is higher along corrugations."
    },
    "Truss Core": {
        "rel_density_factor": 0.055,
        "shear_mod_factor":   0.050,
        "note": "Triangulated struts transfer shear loads efficiently between face sheets for lightweight structural panels."
    },
    "Foam Core": {
        "rel_density_factor": 0.085,
        "shear_mod_factor":   0.025,
        "note": "Continuous foam supports face sheets uniformly and resists local wrinkling, though often heavier per unit stiffness."
    },
}

G_ACCEL = 9.81

# ==========================================
# 2. SANDWICH PANEL STIFFNESS & WEIGHT MODEL
# ==========================================
def calc_sandwich_metrics(face_mat_name, core_pattern, panel_w_mm, panel_l_mm, t_face_mm, t_core_mm, pitch_mm):
    f_mat = FACE_MATERIALS[face_mat_name]
    c_pat = CORE_PATTERNS[core_pattern]

    b = panel_w_mm * 1e-3
    L = panel_l_mm * 1e-3
    tf = t_face_mm * 1e-3
    tc = t_core_mm * 1e-3
    p = max(pitch_mm * 1e-3, 2e-3)

    # Effective core density & shear modulus scaled by pattern & pitch
    pitch_scale = float(np.clip((12e-3 / p) ** 0.25, 0.75, 1.35))
    rho_core_eff = f_mat["rho"] * c_pat["rel_density_factor"] * pitch_scale
    G_core_eff = f_mat["E"] * 0.35 * c_pat["shear_mod_factor"] * pitch_scale

    # Two face sheets + core mass
    m_faces = 2.0 * (f_mat["rho"] * b * L * tf)
    m_core = rho_core_eff * b * L * tc
    mass_kg = m_faces + m_core
    weight_N = mass_kg * G_ACCEL

    # Sandwich bending stiffness per unit width / full width b:
    # EI_eq ~ E_f * (b * tf^3 / 6 + b * tf * (tc + tf)^2 / 2) + G_core * b * tc^3 / 12
    d_cent = tc + tf
    EI_faces = f_mat["E"] * (2.0 * (b * tf**3 / 12.0) + 2.0 * b * tf * (0.5 * d_cent)**2)
    EI_core = G_core_eff * (b * tc**3 / 12.0)
    EI_eq = EI_faces + EI_core

    stiffness_per_weight = EI_eq / max(weight_N, 1e-9)

    return {
        "mass_kg": mass_kg,
        "mass_g": mass_kg * 1000.0,
        "weight_N": weight_N,
        "EI_eq": EI_eq,
        "stiffness_per_weight": stiffness_per_weight,
        "rho_core_eff": rho_core_eff,
    }

# ==========================================
# 3. 2D PANEL CROSS-SECTION + STIFFNESS VS WEIGHT PLOT
# ==========================================
def draw_panel_2d_and_stiffness_vs_weight(face_mat_name, core_pattern, panel_w_mm, panel_l_mm, t_face_mm, t_core_mm, pitch_mm):
    cur = calc_sandwich_metrics(face_mat_name, core_pattern, panel_w_mm, panel_l_mm, t_face_mm, t_core_mm, pitch_mm)

    plt.style.use("seaborn-v0_8-whitegrid")
    fig, (ax_2d, ax_sw) = plt.subplots(1, 2, figsize=(13.5, 4.6), gridspec_kw={"width_ratios": [1.05, 1.35]})

    # --- LEFT: 2D Cross-Section of Sandwich Panel ---
    draw_len = 160.0  # mm visual segment
    tf = t_face_mm
    tc = t_core_mm

    # Bottom & Top face sheets
    ax_2d.add_patch(patches.Rectangle((0, 0), draw_len, tf, facecolor="#2563eb", edgecolor="#0f172a", lw=1.2, label="Face Sheets"))
    ax_2d.add_patch(patches.Rectangle((0, tf + tc), draw_len, tf, facecolor="#2563eb", edgecolor="#0f172a", lw=1.2))

    # Core region background
    ax_2d.add_patch(patches.Rectangle((0, tf), draw_len, tc, facecolor="#f1f5f9", edgecolor="#94a3b8", lw=0.8, alpha=0.7))

    # Draw core pattern inside [tf, tf + tc]
    x_grid = np.linspace(0, draw_len, 400)
    if core_pattern == "Honeycomb":
        cell_w = max(pitch_mm, 8.0)
        xs = np.arange(0, draw_len + cell_w, cell_w)
        for x0 in xs[:-1]:
            x1 = min(x0 + cell_w, draw_len)
            xm = 0.5 * (x0 + x1)
            y_bot, y_top = tf, tf + tc
            y_mid = 0.5 * (y_bot + y_top)
            pts = [[x0, y_mid], [x0 + 0.25*(x1-x0), y_top], [x0 + 0.75*(x1-x0), y_top],
                   [x1, y_mid], [x0 + 0.75*(x1-x0), y_bot], [x0 + 0.25*(x1-x0), y_bot], [x0, y_mid]]
            px, py = zip(*pts)
            ax_2d.plot(px, py, color="#f97316", lw=1.5)
    elif core_pattern == "Corrugated (Wave)":
        wave_y = tf + 0.5 * tc + 0.45 * tc * np.sin(2.0 * np.pi * x_grid / max(pitch_mm, 10.0))
        ax_2d.plot(x_grid, wave_y, color="#f97316", lw=1.8, label="Core Pattern")
    elif core_pattern == "Truss Core":
        step = max(pitch_mm, 12.0)
        xs = np.arange(0, draw_len + step, step)
        for i in range(len(xs) - 1):
            x0, x1 = xs[i], xs[i+1]
            y_bot, y_top = tf, tf + tc
            if i % 2 == 0:
                ax_2d.plot([x0, x1], [y_bot, y_top], color="#f97316", lw=1.6)
            else:
                ax_2d.plot([x0, x1], [y_top, y_bot], color="#f97316", lw=1.6)
        ax_2d.plot(xs, np.full_like(xs, tf + tc), color="#f97316", lw=1.0, alpha=0.6)
        ax_2d.plot(xs, np.full_like(xs, tf), color="#f97316", lw=1.0, alpha=0.6)
    else:  # "Foam Core"
        rng = np.random.default_rng(42)
        fx = rng.uniform(4.0, draw_len - 4.0, size=65)
        fy = rng.uniform(tf + 1.0, tf + tc - 1.0, size=65)
        ax_2d.scatter(fx, fy, s=10, color="#64748b", alpha=0.65, label="Foam Cells")

    ax_2d.set_xlim(-5, draw_len + 5)
    ax_2d.set_ylim(-0.25 * (2*tf + tc), 1.25 * (2*tf + tc))
    ax_2d.set_aspect("equal")
    ax_2d.set_title(f"2D Panel Cross-Section — {core_pattern}", fontsize=11, fontweight="bold", pad=10)
    ax_2d.set_xlabel("Section Segment [mm] (Schematic 2D)", fontsize=9)
    ax_2d.set_ylabel("Through-Thickness [mm]", fontsize=9)
    ax_2d.legend(loc="upper right", fontsize=8, frameon=True, facecolor="white")

    # --- RIGHT: Stiffness vs. Weight Graph across Core Thickness & Patterns ---
    t_core_sweep = np.linspace(4.0, 40.0, 40)
    pattern_colors = {
        "Honeycomb": "#2563eb",
        "Corrugated (Wave)": "#16a34a",
        "Truss Core": "#9333ea",
        "Foam Core": "#64748b",
    }

    for pat_name, p_col in pattern_colors.items():
        w_list, ei_list = [], []
        for tc_val in t_core_sweep:
            m = calc_sandwich_metrics(face_mat_name, pat_name, panel_w_mm, panel_l_mm, t_face_mm, tc_val, pitch_mm)
            w_list.append(m["weight_N"])
            ei_list.append(m["EI_eq"])

        lw_line = 2.8 if pat_name == core_pattern else 1.7
        alpha_line = 1.0 if pat_name == core_pattern else 0.72
        ax_sw.plot(w_list, ei_list, color=p_col, lw=lw_line, alpha=alpha_line, label=f"{pat_name}")

    # Highlight currently selected operating point
    ax_sw.scatter([cur["weight_N"]], [cur["EI_eq"]], color="#dc2626", s=85, zorder=6, edgecolor="white", lw=1.2)
    ax_sw.annotate(
        f"Selected ({core_pattern})\nWeight: {cur['weight_N']:.2f} N ({cur['mass_g']:.0f} g)\nEI_eq: {cur['EI_eq']:.1f} N·m²",
        xy=(cur["weight_N"], cur["EI_eq"]),
        xytext=(12, 10), textcoords="offset points",
        fontsize=8.5, fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.35", facecolor="#fff7ed", edgecolor="#f97316"),
        arrowprops=dict(arrowstyle="->", color="#dc2626", lw=1.2)
    )

    ax_sw.set_title(f"Stiffness vs. Weight — {face_mat_name}", fontsize=11, fontweight="bold", pad=10)
    ax_sw.set_xlabel("Panel Weight [N] (sweeping core thickness 4–40 mm)", fontsize=9.5)
    ax_sw.set_ylabel("Bending Stiffness EI_eq [N·m²]", fontsize=9.5)
    ax_sw.legend(loc="upper left", fontsize=8.5, frameon=True, facecolor="white", edgecolor="#cbd5e1")

    plt.tight_layout()
    plt.show()

    # Summary Card
    display(HTML(f"""
    <div style="
        width: 100%;
        box-sizing: border-box;
        background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);
        border: 1px solid #334155;
        border-left: 4px solid #38bdf8;
        border-radius: 8px;
        padding: 13px 17px;
        color: #f8fafc;
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
        margin-top: 8px;
    ">
        <div style="font-size:13px;margin-bottom:5px;">
            <b>Selected Core:</b> <span style="color:#7dd3fc;font-weight:700;">{core_pattern}</span> &nbsp;|&nbsp;
            <b>Mass:</b> {cur['mass_g']:.1f} g ({cur['weight_N']:.2f} N) &nbsp;|&nbsp;
            <b>Bending Stiffness (EI_eq):</b> {cur['EI_eq']:.1f} N·m² &nbsp;|&nbsp;
            <b>Stiffness / Weight:</b> {cur['stiffness_per_weight']:.1f} N·m²/N
        </div>
        <div style="font-size:12.5px;color:#cbd5e1;line-height:1.5;">
            {CORE_PATTERNS[core_pattern]['note']}
        </div>
    </div>
    """))

# ==========================================
# 4. INTERACTIVE ENGLISH UI
# ==========================================
lbl_style = {"description_width": "145px"}
ctrl_layout = widgets.Layout(width="340px")

w_fmat  = widgets.Dropdown(options=list(FACE_MATERIALS.keys()), value="Aluminum 6061-T6", description="Face Material:", style=lbl_style, layout=ctrl_layout)
w_core  = widgets.Dropdown(options=list(CORE_PATTERNS.keys()), value="Honeycomb", description="Core Pattern:", style=lbl_style, layout=ctrl_layout)
w_tf    = widgets.FloatSlider(value=1.0, min=0.3, max=4.0, step=0.1, description="Face Thick. [mm]:", readout_format=".1f", style=lbl_style, layout=ctrl_layout)
w_tc    = widgets.FloatSlider(value=15.0, min=4.0, max=40.0, step=1.0, description="Core Thick. [mm]:", readout_format=".0f", style=lbl_style, layout=ctrl_layout)
w_pitch = widgets.FloatSlider(value=12.0, min=6.0, max=30.0, step=1.0, description="Cell/Pitch [mm]:", readout_format=".0f", style=lbl_style, layout=ctrl_layout)
w_len   = widgets.FloatSlider(value=400.0, min=150.0, max=1000.0, step=25.0, description="Panel Length [mm]:", readout_format=".0f", style=lbl_style, layout=ctrl_layout)
w_wid   = widgets.FloatSlider(value=200.0, min=100.0, max=600.0, step=20.0, description="Panel Width [mm]:", readout_format=".0f", style=lbl_style, layout=ctrl_layout)

btn_update = widgets.Button(
    description="Update Panel & Graph",
    button_style="primary",
    icon="refresh",
    layout=widgets.Layout(width="220px", height="36px")
)

out_panel = widgets.Output(layout=widgets.Layout(width="100%"))

def refresh_panel(*args):
    with out_panel:
        clear_output(wait=True)
        draw_panel_2d_and_stiffness_vs_weight(
            w_fmat.value, w_core.value, w_wid.value, w_len.value,
            w_tf.value, w_tc.value, w_pitch.value
        )

btn_update.on_click(refresh_panel)
for c in [w_fmat, w_core, w_tf, w_tc, w_pitch, w_len, w_wid]:
    c.observe(refresh_panel, names="value")

header_ui = widgets.HTML("""
    <div style="
        width:100%;box-sizing:border-box;
        background:linear-gradient(135deg,#0f172a 0%,#1e293b 100%);
        border:1px solid #334155;border-left:4px solid #38bdf8;
        border-radius:8px;padding:12px 16px;margin-bottom:10px;
        font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;
    ">
        <div style="font-size:17px;font-weight:700;color:#f8fafc;">Sandwich Panels — Core Patterns &amp; Stiffness vs. Weight</div>
        <div style="font-size:12px;color:#94a3b8;margin-top:2px;">
            Inspect 2D core geometry and plot bending stiffness against panel weight across core patterns
        </div>
    </div>
""")

controls_ui = widgets.VBox([
    widgets.HBox([
        widgets.VBox([w_fmat, w_core, w_tf, w_tc], layout=widgets.Layout(margin="0 28px 0 0", gap="8px")),
        widgets.VBox([w_pitch, w_len, w_wid, btn_update], layout=widgets.Layout(gap="8px"))
    ], layout=widgets.Layout(align_items="flex-start", flex_flow="row wrap"))
], layout=widgets.Layout(
    width="100%", padding="14px 16px", border="1px solid #334155",
    border_radius="8px", background_color="#1e293b", margin="0 0 12px 0"
))

display(header_ui, controls_ui, out_panel)
refresh_panel()

HTML(value='\n    <div style="\n        width:100%;box-sizing:border-box;\n        background:linear-gradient(…

Output(layout=Layout(width='100%'))